# Multi-Label Style Classification Training

## 1. Objective

The objective of this notebook is to train a multi-label style classifier for the fashion recommendation system.

The previous experiment created a multi-label training CSV by manually reviewing ambiguous clothing items. This allows some images to have more than one valid aesthetic label, for example:

- sporty + streetwear
- gothic + streetwear
- formal + gothic

This notebook tests the teacher feedback directly by changing the learning setup from single-label classification to multi-label classification.

Instead of forcing the model to choose only one style, the model will learn to predict each style independently.

The main research question is:

**Does training the style classifier as a multi-label model improve the handling of overlapping fashion aesthetics compared with the current single-label MobileNetV3 model?**

## 2. Project Context

The current selected style model is MobileNetV3 Large trained with single-label classification.

In the current setup, each image is treated as belonging to one style only:

```text
formal OR gothic OR sporty OR streetwear
```

However, fashion items can realistically belong to more than one style at the same time. For example, a black graphic t-shirt may be both gothic and streetwear, while a track jacket may be both sporty and streetwear.

The previous notebook used error analysis and manual review to create a new training file:

```text
../results/style_multilabel_error_analysis/style_multilabel_train_labels.csv
```

## 3. Experiment Design

This experiment changes the style classification learning setup.

The main changes are:

| Part | Previous single-label setup | New multi-label setup |
|---|---|---|
| Target format | one class index | multi-hot vector |
| Loss function | CrossEntropyLoss | BCEWithLogitsLoss |
| Inference activation | softmax | sigmoid |
| Output meaning | one winning style | independent score per style |

The model architecture stays MobileNetV3 Large. This keeps the comparison focused on the learning paradigm instead of changing the architecture again.

The model will be evaluated in two ways:

1. **Single-label compatibility evaluation**  
   The highest sigmoid score is treated as the main style prediction. This makes the result comparable with the previous model.

2. **Multi-label behaviour evaluation**  
   All styles above a threshold are treated as predicted styles. This checks whether the model can produce combined aesthetic outputs.

## 4. Imports and setup

In [1]:
import os
import time
import copy
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

# Device setup
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using device: CUDA")
    print("GPU:", torch.cuda.get_device_name(0))
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS")
else:
    device = torch.device("cpu")
    print("Using device: CPU")

print("Torch version:", torch.__version__)

Using device: CUDA
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Torch version: 2.11.0+cu130


## 5. Paths

In [2]:
# Input paths
multilabel_train_labels_path = Path("../results/style_multilabel_error_analysis/style_multilabel_train_labels.csv")

split_style_root = Path("../dataset/split_style")
real_world_root = Path("../dataset/real_world_test")

# Baseline model for later comparison
single_label_model_path = Path("../models/style_mobilenet_v3_large_architecture_comparison.pth")

# Output paths
results_dir = Path("../results/style_multilabel_training")
results_dir.mkdir(parents=True, exist_ok=True)

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

print("Multi-label train CSV exists:", multilabel_train_labels_path.exists())
print("split_style exists:", split_style_root.exists())
print("real_world_test exists:", real_world_root.exists())
print("Single-label baseline model exists:", single_label_model_path.exists())
print("Results folder:", results_dir)

Multi-label train CSV exists: True
split_style exists: True
real_world_test exists: True
Single-label baseline model exists: True
Results folder: ..\results\style_multilabel_training


## 6. Load Multi-Label Training Labels

The training data comes from the CSV created in the previous experiment.

Each row contains the image path and four label columns:

- label_formal
- label_gothic
- label_sporty
- label_streetwear

An image can have one label or multiple labels.

In [3]:
train_labels_df = pd.read_csv(multilabel_train_labels_path)

class_names = ["formal", "gothic", "sporty", "streetwear"]
label_columns = [f"label_{class_name}" for class_name in class_names]

print("Rows:", len(train_labels_df))
print("Columns:", train_labels_df.columns.tolist())

display(train_labels_df.head())

Rows: 660
Columns: ['source', 'split', 'filename', 'image_path', 'true_style', 'was_manually_reviewed', 'review_decision', 'manual_label_names', 'manual_notes', 'label_formal', 'label_gothic', 'label_sporty', 'label_streetwear', 'label_count']


,source,split,filename,image_path,true_style,was_manually_reviewed,review_decision,manual_label_names,manual_notes,label_formal,label_gothic,label_sporty,label_streetwear,label_count
0,split_style_train,train,jacket_formal_jacket_001.png,..\dataset\split_style\train\formal\jacket_for...,formal,False,not_reviewed,NaN,NaN,1,0,0,0,1
1,split_style_train,train,jacket_formal_jacket_003.png,..\dataset\split_style\train\formal\jacket_for...,formal,False,not_reviewed,NaN,NaN,1,0,0,0,1
2,split_style_train,train,jacket_formal_jacket_004.png,..\dataset\split_style\train\formal\jacket_for...,formal,False,not_reviewed,NaN,NaN,1,0,0,0,1
3,split_style_train,train,jacket_formal_jacket_005.png,..\dataset\split_style\train\formal\jacket_for...,formal,False,not_reviewed,NaN,NaN,1,0,0,0,1
4,split_style_train,train,jacket_formal_jacket_006.png,..\dataset\split_style\train\formal\jacket_for...,formal,False,not_reviewed,NaN,NaN,1,0,0,0,1


## 7. Validate training labels

In [4]:
missing_label_columns = [
    col for col in label_columns
    if col not in train_labels_df.columns
]

if missing_label_columns:
    raise ValueError(f"Missing label columns: {missing_label_columns}")

# Make sure labels are integers 0 or 1
for col in label_columns:
    train_labels_df[col] = train_labels_df[col].astype(int)

train_labels_df["label_count"] = train_labels_df[label_columns].sum(axis=1)

print("Label count distribution:")
display(train_labels_df["label_count"].value_counts().sort_index())

print("Label totals:")
label_totals = train_labels_df[label_columns].sum().reset_index()
label_totals.columns = ["label", "count"]
display(label_totals)

zero_label_rows = train_labels_df[train_labels_df["label_count"] == 0]

print("Rows with zero labels:", len(zero_label_rows))

if len(zero_label_rows) > 0:
    display(zero_label_rows[["filename", "image_path"] + label_columns])

# Check image paths
train_labels_df["image_exists"] = train_labels_df["image_path"].apply(
    lambda path: Path(path).exists()
)

missing_images = train_labels_df[~train_labels_df["image_exists"]]

print("Missing images:", len(missing_images))

if len(missing_images) > 0:
    display(missing_images[["filename", "image_path"]].head(20))

Label count distribution:


label_count
1    604
2     56
Name: count, dtype: int64

Label totals:


,label,count
0,label_formal,169
1,label_gothic,147
2,label_sporty,182
3,label_streetwear,218


Rows with zero labels: 0
Missing images: 0


## 8. Dataset and Image Transforms

The training dataset uses the multi-label CSV.

The validation, curated test, and real-world test sets still come from the original folder structure. These evaluation sets are converted into the same multi-hot format, but each evaluation image has one original label.

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


class MultiLabelStyleDataset(Dataset):
    def __init__(self, dataframe, class_names, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.class_names = class_names
        self.label_columns = [f"label_{class_name}" for class_name in class_names]
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = Path(row["image_path"])
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[self.label_columns].values.astype(np.float32),
            dtype=torch.float32
        )

        return image, labels

## 9. Create evaluation dataframes

In [6]:
valid_extensions = (".jpg", ".jpeg", ".png", ".webp")

def one_hot_labels_for_style(style, class_names):
    return {
        f"label_{class_name}": 1 if class_name == style else 0
        for class_name in class_names
    }


def collect_split_style_dataframe(split_style_root, split_name, class_names):
    rows = []
    split_dir = split_style_root / split_name

    for style in sorted(os.listdir(split_dir)):
        style_path = split_dir / style

        if not style_path.is_dir():
            continue

        if style not in class_names:
            print(f"Skipping unknown style: {style}")
            continue

        for filename in sorted(os.listdir(style_path)):
            image_path = style_path / filename

            if image_path.is_file() and filename.lower().endswith(valid_extensions):
                row = {
                    "source": f"split_style_{split_name}",
                    "split": split_name,
                    "filename": filename,
                    "image_path": str(image_path),
                    "true_style": style
                }

                row.update(one_hot_labels_for_style(style, class_names))
                rows.append(row)

    return pd.DataFrame(rows)


def collect_real_world_dataframe(real_world_root, class_names):
    rows = []

    for style in sorted(os.listdir(real_world_root)):
        style_path = real_world_root / style

        if not style_path.is_dir():
            continue

        if style not in class_names:
            print(f"Skipping unknown style: {style}")
            continue

        for item_type in sorted(os.listdir(style_path)):
            type_path = style_path / item_type

            if not type_path.is_dir():
                continue

            for filename in sorted(os.listdir(type_path)):
                image_path = type_path / filename

                if image_path.is_file() and filename.lower().endswith(valid_extensions):
                    row = {
                        "source": "real_world_test",
                        "split": "real_world_test",
                        "filename": filename,
                        "image_path": str(image_path),
                        "true_style": style,
                        "type": item_type
                    }

                    row.update(one_hot_labels_for_style(style, class_names))
                    rows.append(row)

    return pd.DataFrame(rows)


val_df = collect_split_style_dataframe(split_style_root, "val", class_names)
curated_test_df = collect_split_style_dataframe(split_style_root, "test", class_names)
real_world_df = collect_real_world_dataframe(real_world_root, class_names)

print("Training rows:", len(train_labels_df))
print("Validation rows:", len(val_df))
print("Curated test rows:", len(curated_test_df))
print("Real-world test rows:", len(real_world_df))

display(val_df.head())

Training rows: 660
Validation rows: 120
Curated test rows: 120
Real-world test rows: 80


,source,split,filename,image_path,true_style,label_formal,label_gothic,label_sporty,label_streetwear
0,split_style_val,val,jacket_formal_jacket_002.png,..\dataset\split_style\val\formal\jacket_forma...,formal,1,0,0,0
1,split_style_val,val,jacket_formal_jacket_012.png,..\dataset\split_style\val\formal\jacket_forma...,formal,1,0,0,0
2,split_style_val,val,jacket_formal_jacket_018.png,..\dataset\split_style\val\formal\jacket_forma...,formal,1,0,0,0
3,split_style_val,val,jacket_formal_jacket_021.png,..\dataset\split_style\val\formal\jacket_forma...,formal,1,0,0,0
4,split_style_val,val,jacket_formal_jacket_025.png,..\dataset\split_style\val\formal\jacket_forma...,formal,1,0,0,0


## 10. Create datasets and dataloaders

In [7]:
batch_size = 32

train_dataset = MultiLabelStyleDataset(
    dataframe=train_labels_df,
    class_names=class_names,
    transform=train_transform
)

val_dataset = MultiLabelStyleDataset(
    dataframe=val_df,
    class_names=class_names,
    transform=eval_transform
)

curated_test_dataset = MultiLabelStyleDataset(
    dataframe=curated_test_df,
    class_names=class_names,
    transform=eval_transform
)

real_world_dataset = MultiLabelStyleDataset(
    dataframe=real_world_df,
    class_names=class_names,
    transform=eval_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

curated_test_loader = DataLoader(
    curated_test_dataset,
    batch_size=batch_size,
    shuffle=False
)

real_world_loader = DataLoader(
    real_world_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Curated test batches:", len(curated_test_loader))
print("Real-world batches:", len(real_world_loader))

Train batches: 21
Validation batches: 4
Curated test batches: 4
Real-world batches: 3


## 11. Build Multi-Label MobileNetV3 Model

This experiment keeps the same architecture as the selected style classifier: MobileNetV3 Large.

The difference is not the architecture, but the learning setup. The model still outputs four values, one for each style, but these outputs are now treated independently.

This means the model can learn that an image may belong to more than one aesthetic at the same time.

In [8]:
def freeze_model_parameters(model):
    for param in model.parameters():
        param.requires_grad = False


def build_multilabel_mobilenet_v3(num_classes):
    weights = models.MobileNet_V3_Large_Weights.DEFAULT
    model = models.mobilenet_v3_large(weights=weights)

    # Keep the same transfer learning approach as previous experiments
    freeze_model_parameters(model)

    in_features = model.classifier[-1].in_features
    model.classifier[-1] = nn.Linear(in_features, num_classes)

    return model


model = build_multilabel_mobilenet_v3(num_classes=len(class_names))
model = model.to(device)

dummy_input = torch.randn(1, 3, 224, 224).to(device)

with torch.no_grad():
    dummy_output = model(dummy_input)

print("Output shape:", dummy_output.shape)
print("Expected shape: [1, 4]")

Output shape: torch.Size([1, 4])
Expected shape: [1, 4]


## 12. Multi-Label Prediction Helpers

In the single-label model, softmax was used because one class had to win.

In this multi-label setup, sigmoid is used instead. This gives an independent score for each style.

For evaluation, I use two views:

1. **Top-1 prediction**: the style with the highest sigmoid score. This allows comparison with the previous single-label model.
2. **Threshold prediction**: all styles above a selected threshold. This shows the multi-label behaviour.

In [9]:
sigmoid_threshold = 0.50


def logits_to_sigmoid_scores(logits):
    return torch.sigmoid(logits)


def get_top1_predictions_from_scores(scores):
    return torch.argmax(scores, dim=1)


def get_threshold_predictions_from_scores(scores, threshold=0.50):
    return (scores >= threshold).int()

## 13. Training Function

The model is trained using `BCEWithLogitsLoss`.

This loss function is suitable for multi-label classification because each output class is treated independently. An image can therefore have multiple positive labels.

In [10]:
def train_multilabel_model(
    model,
    train_loader,
    val_loader,
    num_epochs=10,
    learning_rate=0.001
):
    model = model.to(device)

    criterion = nn.BCEWithLogitsLoss()

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate
    )

    best_model_state = copy.deepcopy(model.state_dict())
    best_val_top1_accuracy = 0.0

    history = []

    start_time = time.time()

    for epoch in range(num_epochs):
        print("=" * 80)
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print("=" * 80)

        # Training phase
        model.train()

        train_loss = 0.0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            train_total += labels.size(0)

        train_epoch_loss = train_loss / train_total

        # Validation phase
        model.eval()

        val_loss = 0.0
        val_total = 0

        all_val_true_top1 = []
        all_val_pred_top1 = []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                logits = model(images)
                loss = criterion(logits, labels)

                scores = torch.sigmoid(logits)

                true_top1 = torch.argmax(labels, dim=1)
                pred_top1 = torch.argmax(scores, dim=1)

                all_val_true_top1.extend(true_top1.cpu().numpy())
                all_val_pred_top1.extend(pred_top1.cpu().numpy())

                val_loss += loss.item() * images.size(0)
                val_total += labels.size(0)

        val_epoch_loss = val_loss / val_total
        val_top1_accuracy = accuracy_score(all_val_true_top1, all_val_pred_top1)
        val_top1_macro_f1 = f1_score(
            all_val_true_top1,
            all_val_pred_top1,
            average="macro",
            zero_division=0
        )

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_epoch_loss,
            "val_loss": val_epoch_loss,
            "val_top1_accuracy": val_top1_accuracy,
            "val_top1_macro_f1": val_top1_macro_f1
        })

        print(f"Train loss:       {train_epoch_loss:.4f}")
        print(f"Validation loss:  {val_epoch_loss:.4f}")
        print(f"Val top-1 acc:    {val_top1_accuracy:.4f}")
        print(f"Val top-1 F1:     {val_top1_macro_f1:.4f}")

        if val_top1_accuracy > best_val_top1_accuracy:
            best_val_top1_accuracy = val_top1_accuracy
            best_model_state = copy.deepcopy(model.state_dict())
            print("Best model updated.")

    total_training_time = time.time() - start_time

    model.load_state_dict(best_model_state)

    history_df = pd.DataFrame(history)

    print("\nTraining finished.")
    print("Best validation top-1 accuracy:", best_val_top1_accuracy)
    print("Training time:", total_training_time)

    return model, history_df, best_val_top1_accuracy, total_training_time

## 14. Train the Multi-Label Model

The model is trained for 10 epochs, using the same basic setup as the previous architecture comparison.

This first experiment keeps the feature extractor frozen and trains only the classifier head. This keeps the experiment controlled and close to the earlier MobileNetV3 setup.

In [11]:
num_epochs = 10
learning_rate = 0.001

model = build_multilabel_mobilenet_v3(num_classes=len(class_names))

trained_model, history_df, best_val_top1_accuracy, training_time = train_multilabel_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=num_epochs,
    learning_rate=learning_rate
)

history_df

Epoch 1/10
Train loss:       0.5948
Validation loss:  0.5350
Val top-1 acc:    0.6000
Val top-1 F1:     0.5838
Best model updated.
Epoch 2/10
Train loss:       0.4630
Validation loss:  0.4613
Val top-1 acc:    0.6833
Val top-1 F1:     0.6717
Best model updated.
Epoch 3/10
Train loss:       0.3971
Validation loss:  0.4127
Val top-1 acc:    0.7500
Val top-1 F1:     0.7455
Best model updated.
Epoch 4/10
Train loss:       0.3567
Validation loss:  0.3772
Val top-1 acc:    0.7583
Val top-1 F1:     0.7545
Best model updated.
Epoch 5/10
Train loss:       0.3286
Validation loss:  0.3511
Val top-1 acc:    0.7833
Val top-1 F1:     0.7747
Best model updated.
Epoch 6/10
Train loss:       0.3022
Validation loss:  0.3277
Val top-1 acc:    0.7667
Val top-1 F1:     0.7611
Epoch 7/10
Train loss:       0.2927
Validation loss:  0.3116
Val top-1 acc:    0.7750
Val top-1 F1:     0.7706
Epoch 8/10
Train loss:       0.2764
Validation loss:  0.3017
Val top-1 acc:    0.7917
Val top-1 F1:     0.7879
Best model u

,epoch,train_loss,val_loss,val_top1_accuracy,val_top1_macro_f1
0,1,0.594800,0.535014,0.600000,0.583766
1,2,0.462984,0.461306,0.683333,0.671710
2,3,0.397111,0.412699,0.750000,0.745531
3,4,0.356675,0.377237,0.758333,0.754495
4,5,0.328554,0.351109,0.783333,0.774654
5,6,0.302223,0.327672,0.766667,0.761142
6,7,0.292675,0.311631,0.775000,0.770577
7,8,0.276352,0.301719,0.791667,0.787899
8,9,0.262362,0.289917,0.791667,0.787957
9,10,0.247864,0.282554,0.800000,0.797006


## 15. Save Training History

The training history is saved so the learning behaviour can be documented and compared later.

In [12]:
history_path = results_dir / "multilabel_mobilenetv3_training_history.csv"

history_df.to_csv(history_path, index=False)

print("Saved training history to:", history_path)

Saved training history to: ..\results\style_multilabel_training\multilabel_mobilenetv3_training_history.csv


## 16. Evaluation Function

After training, the model is evaluated on the curated test set and the real-world test set.

Because this is a multi-label model, I evaluate it in two ways:

1. Top-1 evaluation, where the highest sigmoid score is treated as the main predicted style.
2. Threshold evaluation, where every style above the threshold is treated as a predicted style.

The top-1 evaluation makes the result comparable with the previous single-label model.

In [13]:
def evaluate_multilabel_model(model, data_loader, class_names, threshold=0.50):
    model = model.to(device)
    model.eval()

    all_true_labels = []
    all_scores = []
    all_pred_labels_threshold = []

    all_true_top1 = []
    all_pred_top1 = []

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            scores = torch.sigmoid(logits)

            pred_threshold = (scores >= threshold).int()

            true_top1 = torch.argmax(labels, dim=1)
            pred_top1 = torch.argmax(scores, dim=1)

            all_true_labels.extend(labels.cpu().numpy())
            all_scores.extend(scores.cpu().numpy())
            all_pred_labels_threshold.extend(pred_threshold.cpu().numpy())

            all_true_top1.extend(true_top1.cpu().numpy())
            all_pred_top1.extend(pred_top1.cpu().numpy())

    all_true_labels = np.array(all_true_labels)
    all_scores = np.array(all_scores)
    all_pred_labels_threshold = np.array(all_pred_labels_threshold)

    top1_accuracy = accuracy_score(all_true_top1, all_pred_top1)
    top1_macro_f1 = f1_score(
        all_true_top1,
        all_pred_top1,
        average="macro",
        zero_division=0
    )

    report_text = classification_report(
        all_true_top1,
        all_pred_top1,
        target_names=class_names,
        zero_division=0
    )

    report_dict = classification_report(
        all_true_top1,
        all_pred_top1,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    cm = confusion_matrix(all_true_top1, all_pred_top1)

    predicted_label_count = all_pred_labels_threshold.sum(axis=1)

    multilabel_summary = {
        "avg_predicted_labels": float(predicted_label_count.mean()),
        "single_label_predictions": int((predicted_label_count == 1).sum()),
        "multi_label_predictions": int((predicted_label_count > 1).sum()),
        "zero_label_predictions": int((predicted_label_count == 0).sum())
    }

    return {
        "top1_accuracy": top1_accuracy,
        "top1_macro_f1": top1_macro_f1,
        "classification_report_text": report_text,
        "classification_report_dict": report_dict,
        "confusion_matrix": cm,
        "true_labels": all_true_labels,
        "scores": all_scores,
        "threshold_predictions": all_pred_labels_threshold,
        "true_top1": all_true_top1,
        "pred_top1": all_pred_top1,
        "multilabel_summary": multilabel_summary
    }

## 17. Evaluate on Test Sets

The trained multi-label model is evaluated on the curated test set and the real-world test set.

The real-world test set is the most important result, because it represents the type of user input the final application should handle.

In [14]:
threshold = 0.50

curated_eval = evaluate_multilabel_model(
    model=trained_model,
    data_loader=curated_test_loader,
    class_names=class_names,
    threshold=threshold
)

real_world_eval = evaluate_multilabel_model(
    model=trained_model,
    data_loader=real_world_loader,
    class_names=class_names,
    threshold=threshold
)

print("Curated test top-1 accuracy:", curated_eval["top1_accuracy"])
print("Curated test top-1 macro F1:", curated_eval["top1_macro_f1"])
print("Curated multi-label summary:")
print(curated_eval["multilabel_summary"])

print("\nReal-world test top-1 accuracy:", real_world_eval["top1_accuracy"])
print("Real-world test top-1 macro F1:", real_world_eval["top1_macro_f1"])
print("Real-world multi-label summary:")
print(real_world_eval["multilabel_summary"])

Curated test top-1 accuracy: 0.8
Curated test top-1 macro F1: 0.7968464130958112
Curated multi-label summary:
{'avg_predicted_labels': 1.0333333333333334, 'single_label_predictions': 97, 'multi_label_predictions': 13, 'zero_label_predictions': 10}

Real-world test top-1 accuracy: 0.625
Real-world test top-1 macro F1: 0.6251858204688393
Real-world multi-label summary:
{'avg_predicted_labels': 0.6875, 'single_label_predictions': 47, 'multi_label_predictions': 4, 'zero_label_predictions': 29}


## 18. Classification Reports

In [15]:
print("Curated test classification report:")
print(curated_eval["classification_report_text"])

print("Real-world test classification report:")
print(real_world_eval["classification_report_text"])

Curated test classification report:
              precision    recall  f1-score   support

      formal       0.84      0.90      0.87        30
      gothic       0.93      0.93      0.93        30
      sporty       0.65      0.80      0.72        30
  streetwear       0.81      0.57      0.67        30

    accuracy                           0.80       120
   macro avg       0.81      0.80      0.80       120
weighted avg       0.81      0.80      0.80       120

Real-world test classification report:
              precision    recall  f1-score   support

      formal       0.58      0.70      0.64        20
      gothic       0.92      0.60      0.73        20
      sporty       0.80      0.40      0.53        20
  streetwear       0.48      0.80      0.60        20

    accuracy                           0.62        80
   macro avg       0.70      0.62      0.63        80
weighted avg       0.70      0.62      0.63        80



## 19. Save Evaluation Results

In [16]:
evaluation_summary = {
    "model": "multilabel_mobilenet_v3_large",
    "threshold": threshold,
    "best_val_top1_accuracy": best_val_top1_accuracy,
    "curated_top1_accuracy": curated_eval["top1_accuracy"],
    "curated_top1_macro_f1": curated_eval["top1_macro_f1"],
    "curated_avg_predicted_labels": curated_eval["multilabel_summary"]["avg_predicted_labels"],
    "curated_single_label_predictions": curated_eval["multilabel_summary"]["single_label_predictions"],
    "curated_multi_label_predictions": curated_eval["multilabel_summary"]["multi_label_predictions"],
    "curated_zero_label_predictions": curated_eval["multilabel_summary"]["zero_label_predictions"],
    "real_world_top1_accuracy": real_world_eval["top1_accuracy"],
    "real_world_top1_macro_f1": real_world_eval["top1_macro_f1"],
    "real_world_avg_predicted_labels": real_world_eval["multilabel_summary"]["avg_predicted_labels"],
    "real_world_single_label_predictions": real_world_eval["multilabel_summary"]["single_label_predictions"],
    "real_world_multi_label_predictions": real_world_eval["multilabel_summary"]["multi_label_predictions"],
    "real_world_zero_label_predictions": real_world_eval["multilabel_summary"]["zero_label_predictions"],
    "training_time_seconds": training_time
}

evaluation_summary_df = pd.DataFrame([evaluation_summary])

evaluation_summary_path = results_dir / "multilabel_mobilenetv3_evaluation_summary.csv"

evaluation_summary_df.to_csv(evaluation_summary_path, index=False)

print("Saved evaluation summary to:", evaluation_summary_path)

display(evaluation_summary_df)

Saved evaluation summary to: ..\results\style_multilabel_training\multilabel_mobilenetv3_evaluation_summary.csv


,model,threshold,best_val_top1_accuracy,curated_top1_accuracy,curated_top1_macro_f1,curated_avg_predicted_labels,curated_single_label_predictions,curated_multi_label_predictions,curated_zero_label_predictions,real_world_top1_accuracy,real_world_top1_macro_f1,real_world_avg_predicted_labels,real_world_single_label_predictions,real_world_multi_label_predictions,real_world_zero_label_predictions,training_time_seconds
0,multilabel_mobilenet_v3_large,0.5,0.8,0.8,0.796846,1.033333,97,13,10,0.625,0.625186,0.6875,47,4,29,468.857118


## 20. First Evaluation Interpretation

The first evaluation shows that the multi-label model learns the style task successfully.

The validation accuracy reached 0.8000, and the real-world top-1 accuracy reached 0.6250. This means that when only the highest sigmoid score is used as the main prediction, the model performs similarly to the previous MobileNetV3 style classifier.

However, the threshold-based multi-label output still needs adjustment. With a threshold of 0.50, the model produced many zero-label predictions on the real-world test set. This means that no style reached the threshold for many images.

Because of this, the next step is to test multiple sigmoid thresholds and choose a threshold that gives useful multi-label outputs without creating too many empty predictions.

## 21. Comparison With Previous Single-Label Model

The previous selected MobileNetV3 style classifier achieved:

- 0.8167 curated test accuracy
- 0.8105 curated macro F1-score
- 0.6250 real-world accuracy
- 0.6242 real-world macro F1-score

The multi-label model is compared against these values using top-1 prediction. This is not the full multi-label evaluation yet, but it shows whether the model still works as a normal style classifier.

In [17]:
baseline_comparison = pd.DataFrame([
    {
        "model": "Previous single-label MobileNetV3",
        "curated_accuracy": 0.8167,
        "curated_macro_f1": 0.8105,
        "real_world_accuracy": 0.6250,
        "real_world_macro_f1": 0.6242
    },
    {
        "model": "New multi-label MobileNetV3",
        "curated_accuracy": curated_eval["top1_accuracy"],
        "curated_macro_f1": curated_eval["top1_macro_f1"],
        "real_world_accuracy": real_world_eval["top1_accuracy"],
        "real_world_macro_f1": real_world_eval["top1_macro_f1"]
    }
])

baseline_comparison["curated_accuracy_difference"] = (
    baseline_comparison["curated_accuracy"] - baseline_comparison.loc[0, "curated_accuracy"]
)

baseline_comparison["real_world_accuracy_difference"] = (
    baseline_comparison["real_world_accuracy"] - baseline_comparison.loc[0, "real_world_accuracy"]
)

display(baseline_comparison)

baseline_comparison.to_csv(
    results_dir / "comparison_with_single_label_mobilenetv3.csv",
    index=False
)

,model,curated_accuracy,curated_macro_f1,real_world_accuracy,real_world_macro_f1,curated_accuracy_difference,real_world_accuracy_difference
0,Previous single-label MobileNetV3,0.8167,0.810500,0.625,0.624200,0.0000,0.0
1,New multi-label MobileNetV3,0.8000,0.796846,0.625,0.625186,-0.0167,0.0


## 22. Threshold Sweep

The first threshold test used 0.50, but this created too many zero-label predictions.

In this section, I test multiple sigmoid thresholds. The goal is to find a threshold that allows the model to output multiple styles when appropriate, without returning no style too often.

A lower threshold makes the model more willing to assign multiple aesthetics. A higher threshold makes the model stricter.

In [18]:
def evaluate_threshold_behaviour(eval_result, thresholds):
    rows = []

    scores = eval_result["scores"]
    true_top1 = eval_result["true_top1"]

    for threshold in thresholds:
        threshold_predictions = (scores >= threshold).astype(int)

        predicted_label_count = threshold_predictions.sum(axis=1)

        # Fallback top-1 for accuracy comparison
        pred_top1 = np.argmax(scores, axis=1)

        top1_accuracy = accuracy_score(true_top1, pred_top1)
        top1_macro_f1 = f1_score(
            true_top1,
            pred_top1,
            average="macro",
            zero_division=0
        )

        rows.append({
            "threshold": threshold,
            "top1_accuracy": top1_accuracy,
            "top1_macro_f1": top1_macro_f1,
            "avg_predicted_labels": predicted_label_count.mean(),
            "zero_label_predictions": int((predicted_label_count == 0).sum()),
            "single_label_predictions": int((predicted_label_count == 1).sum()),
            "multi_label_predictions": int((predicted_label_count > 1).sum()),
            "multi_label_rate": float((predicted_label_count > 1).mean()),
            "zero_label_rate": float((predicted_label_count == 0).mean())
        })

    return pd.DataFrame(rows)


thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

curated_threshold_sweep = evaluate_threshold_behaviour(
    curated_eval,
    thresholds
)

real_world_threshold_sweep = evaluate_threshold_behaviour(
    real_world_eval,
    thresholds
)

print("Curated threshold sweep:")
display(curated_threshold_sweep)

print("Real-world threshold sweep:")
display(real_world_threshold_sweep)

curated_threshold_sweep.to_csv(
    results_dir / "curated_threshold_sweep.csv",
    index=False
)

real_world_threshold_sweep.to_csv(
    results_dir / "real_world_threshold_sweep.csv",
    index=False
)

Curated threshold sweep:


,threshold,top1_accuracy,top1_macro_f1,avg_predicted_labels,zero_label_predictions,single_label_predictions,multi_label_predictions,multi_label_rate,zero_label_rate
0,0.20,0.8,0.796846,1.941667,0,34,86,0.716667,0.000000
1,0.25,0.8,0.796846,1.675000,0,50,70,0.583333,0.000000
2,0.30,0.8,0.796846,1.550000,0,62,58,0.483333,0.000000
3,0.35,0.8,0.796846,1.350000,0,81,39,0.325000,0.000000
4,0.40,0.8,0.796846,1.233333,1,92,27,0.225000,0.008333
5,0.45,0.8,0.796846,1.108333,5,98,17,0.141667,0.041667
6,0.50,0.8,0.796846,1.033333,10,97,13,0.108333,0.083333


Real-world threshold sweep:


,threshold,top1_accuracy,top1_macro_f1,avg_predicted_labels,zero_label_predictions,single_label_predictions,multi_label_predictions,multi_label_rate,zero_label_rate
0,0.20,0.625,0.625186,2.1875,0,19,61,0.7625,0.0000
1,0.25,0.625,0.625186,1.8875,0,27,53,0.6625,0.0000
2,0.30,0.625,0.625186,1.6250,2,34,44,0.5500,0.0250
3,0.35,0.625,0.625186,1.3125,8,41,31,0.3875,0.1000
4,0.40,0.625,0.625186,1.0750,13,48,19,0.2375,0.1625
5,0.45,0.625,0.625186,0.9000,22,44,14,0.1750,0.2750
6,0.50,0.625,0.625186,0.6875,29,47,4,0.0500,0.3625


## 23. Choose Practical Threshold

The threshold should not be selected only by accuracy, because top-1 accuracy does not change when only the threshold changes.

Instead, the threshold is selected based on output behaviour:

- avoid many zero-label predictions
- allow some multi-label predictions
- keep the average number of predicted labels reasonable

For the web application, zero-label outputs are not useful. Therefore, the selected threshold should produce very few or no zero-label predictions on realistic images.

In [19]:
# Simple practical selection rule:
# Prefer thresholds with zero-label rate under 10%.
# Among those, choose the highest threshold to avoid over-labeling.

candidate_thresholds = real_world_threshold_sweep[
    real_world_threshold_sweep["zero_label_rate"] <= 0.10
].copy()

if len(candidate_thresholds) > 0:
    selected_threshold = candidate_thresholds["threshold"].max()
else:
    # Fallback if all thresholds still produce too many zero-labels
    selected_threshold = real_world_threshold_sweep.sort_values(
        by=["zero_label_rate", "avg_predicted_labels"],
        ascending=[True, True]
    ).iloc[0]["threshold"]

print("Selected threshold:", selected_threshold)

selected_threshold_summary = real_world_threshold_sweep[
    real_world_threshold_sweep["threshold"] == selected_threshold
]

display(selected_threshold_summary)

Selected threshold: 0.35


,threshold,top1_accuracy,top1_macro_f1,avg_predicted_labels,zero_label_predictions,single_label_predictions,multi_label_predictions,multi_label_rate,zero_label_rate
3,0.35,0.625,0.625186,1.3125,8,41,31,0.3875,0.1


## 24. Apply Selected Threshold With Top-1 Fallback

The selected sigmoid threshold is 0.35.

However, a threshold can still produce zero-label predictions when no style score is high enough. This is not useful for the final application, because the system should always return at least one aesthetic direction.

To solve this, I use a top-1 fallback:

- first, select all styles above the threshold
- if no style passes the threshold, select the highest scoring style anyway

This keeps the multi-label behaviour while avoiding empty outputs.

In [20]:
selected_threshold = 0.35


def apply_threshold_with_top1_fallback(scores, threshold):
    threshold_predictions = (scores >= threshold).astype(int)

    for i in range(len(threshold_predictions)):
        if threshold_predictions[i].sum() == 0:
            top_index = np.argmax(scores[i])
            threshold_predictions[i, top_index] = 1

    return threshold_predictions


def summarize_prediction_counts(predictions):
    predicted_label_count = predictions.sum(axis=1)

    return {
        "avg_predicted_labels": float(predicted_label_count.mean()),
        "zero_label_predictions": int((predicted_label_count == 0).sum()),
        "single_label_predictions": int((predicted_label_count == 1).sum()),
        "multi_label_predictions": int((predicted_label_count > 1).sum()),
        "zero_label_rate": float((predicted_label_count == 0).mean()),
        "multi_label_rate": float((predicted_label_count > 1).mean())
    }


curated_predictions_with_fallback = apply_threshold_with_top1_fallback(
    curated_eval["scores"],
    selected_threshold
)

real_world_predictions_with_fallback = apply_threshold_with_top1_fallback(
    real_world_eval["scores"],
    selected_threshold
)

curated_fallback_summary = summarize_prediction_counts(
    curated_predictions_with_fallback
)

real_world_fallback_summary = summarize_prediction_counts(
    real_world_predictions_with_fallback
)

print("Selected threshold:", selected_threshold)

print("\nCurated predictions with fallback:")
print(curated_fallback_summary)

print("\nReal-world predictions with fallback:")
print(real_world_fallback_summary)

Selected threshold: 0.35

Curated predictions with fallback:
{'avg_predicted_labels': 1.35, 'zero_label_predictions': 0, 'single_label_predictions': 81, 'multi_label_predictions': 39, 'zero_label_rate': 0.0, 'multi_label_rate': 0.325}

Real-world predictions with fallback:
{'avg_predicted_labels': 1.4125, 'zero_label_predictions': 0, 'single_label_predictions': 49, 'multi_label_predictions': 31, 'zero_label_rate': 0.0, 'multi_label_rate': 0.3875}


## 25. Create Prediction Tables

This section creates readable prediction tables for the curated and real-world test sets.

The tables include:

- true style
- predicted main style
- all predicted styles after threshold + fallback
- individual sigmoid scores for each style

These tables are useful for manual inspection and for checking whether the multi-label outputs make sense.

In [21]:
def labels_to_style_names(label_vector, class_names):
    selected_styles = [
        class_names[i]
        for i, value in enumerate(label_vector)
        if value == 1
    ]

    return " + ".join(selected_styles)


def create_prediction_table(source_df, eval_result, fallback_predictions, class_names):
    prediction_df = source_df.reset_index(drop=True).copy()

    scores = eval_result["scores"]
    pred_top1 = eval_result["pred_top1"]

    prediction_df["predicted_main_style"] = [
        class_names[index]
        for index in pred_top1
    ]

    prediction_df["predicted_styles_threshold_fallback"] = [
        labels_to_style_names(row, class_names)
        for row in fallback_predictions
    ]

    prediction_df["predicted_label_count"] = fallback_predictions.sum(axis=1)

    for idx, class_name in enumerate(class_names):
        prediction_df[f"score_{class_name}"] = scores[:, idx]

    prediction_df["top1_correct"] = (
        prediction_df["true_style"] == prediction_df["predicted_main_style"]
    )

    return prediction_df


curated_prediction_df = create_prediction_table(
    curated_test_df,
    curated_eval,
    curated_predictions_with_fallback,
    class_names
)

real_world_prediction_df = create_prediction_table(
    real_world_df,
    real_world_eval,
    real_world_predictions_with_fallback,
    class_names
)

curated_prediction_path = results_dir / "curated_multilabel_predictions_threshold_035.csv"
real_world_prediction_path = results_dir / "real_world_multilabel_predictions_threshold_035.csv"

curated_prediction_df.to_csv(curated_prediction_path, index=False)
real_world_prediction_df.to_csv(real_world_prediction_path, index=False)

print("Saved curated predictions to:", curated_prediction_path)
print("Saved real-world predictions to:", real_world_prediction_path)

display(real_world_prediction_df[
    [
        "filename",
        "true_style",
        "predicted_main_style",
        "predicted_styles_threshold_fallback",
        "predicted_label_count",
        "score_formal",
        "score_gothic",
        "score_sporty",
        "score_streetwear",
        "top1_correct"
    ]
].head(20))

Saved curated predictions to: ..\results\style_multilabel_training\curated_multilabel_predictions_threshold_035.csv
Saved real-world predictions to: ..\results\style_multilabel_training\real_world_multilabel_predictions_threshold_035.csv


,filename,true_style,predicted_main_style,predicted_styles_threshold_fallback,predicted_label_count,score_formal,score_gothic,score_sporty,score_streetwear,top1_correct
0,formal_jacket_rw_001.png,formal,formal,formal,1,0.571103,0.028740,0.143603,0.214055,True
1,formal_jacket_rw_002.png,formal,formal,formal,1,0.906757,0.045820,0.249575,0.192100,True
2,formal_jacket_rw_003.png,formal,streetwear,formal + streetwear,2,0.366350,0.212848,0.257413,0.414322,False
3,formal_jacket_rw_004.png,formal,formal,formal + streetwear,2,0.772820,0.035382,0.298145,0.464679,True
4,formal_jacket_rw_005.png,formal,formal,formal,1,0.527818,0.007305,0.086710,0.079321,True
5,formal_pants_rw_001.png,formal,formal,formal,1,0.648786,0.007470,0.290163,0.161111,True
6,formal_pants_rw_002.png,formal,sporty,sporty,1,0.160273,0.013412,0.536006,0.167923,False
7,formal_pants_rw_003.png,formal,streetwear,streetwear,1,0.321302,0.012702,0.213418,0.357844,False
8,formal_pants_rw_004.png,formal,formal,formal,1,0.701181,0.036439,0.259690,0.208543,True
9,formal_pants_rw_005.png,formal,formal,formal + sporty,2,0.662888,0.021618,0.356717,0.324128,True


## 26. Inspect Multi-Label Real-World Predictions

The real-world test set is inspected because it best represents realistic user uploads.

This section shows only the images where the model predicted more than one style. These are the most important examples for checking whether the multi-label behaviour is useful.

In [22]:
real_world_multilabel_examples = real_world_prediction_df[
    real_world_prediction_df["predicted_label_count"] > 1
].copy()

print("Real-world multi-label predictions:", len(real_world_multilabel_examples))

display(real_world_multilabel_examples[
    [
        "filename",
        "true_style",
        "predicted_main_style",
        "predicted_styles_threshold_fallback",
        "score_formal",
        "score_gothic",
        "score_sporty",
        "score_streetwear",
        "top1_correct"
    ]
].head(30))

Real-world multi-label predictions: 31


,filename,true_style,predicted_main_style,predicted_styles_threshold_fallback,score_formal,score_gothic,score_sporty,score_streetwear,top1_correct
2,formal_jacket_rw_003.png,formal,streetwear,formal + streetwear,0.366350,0.212848,0.257413,0.414322,False
3,formal_jacket_rw_004.png,formal,formal,formal + streetwear,0.772820,0.035382,0.298145,0.464679,True
9,formal_pants_rw_005.png,formal,formal,formal + sporty,0.662888,0.021618,0.356717,0.324128,True
10,formal_shoes_rw_001.png,formal,formal,formal + gothic,0.634688,0.565728,0.139473,0.033961,True
11,formal_shoes_rw_002.png,formal,formal,formal + sporty,0.459374,0.324262,0.360061,0.138171,True
15,formal_tshirt_rw_001.png,formal,formal,formal + sporty,0.549469,0.148040,0.494937,0.198103,True
16,formal_tshirt_rw_002.png,formal,streetwear,formal + streetwear,0.459686,0.063399,0.318647,0.569447,False
17,formal_tshirt_rw_003.png,formal,streetwear,sporty + streetwear,0.338894,0.065589,0.408803,0.615044,False
18,formal_tshirt_rw_004.png,formal,formal,formal + streetwear,0.416448,0.016171,0.238217,0.392594,True
19,formal_tshirt_rw_005.png,formal,streetwear,sporty + streetwear,0.173865,0.043939,0.364971,0.464462,False


## 27. Save Multi-Label Model Checkpoint

The trained multi-label MobileNetV3 model is saved for later use.

The checkpoint stores the model weights, class names, selected threshold, and training metadata. This is needed if the model is later integrated into the recommendation backend.

In [23]:
multilabel_model_path = models_dir / "style_mobilenet_v3_large_multilabel_bce.pth"

checkpoint = {
    "model_state_dict": trained_model.state_dict(),
    "class_names": class_names,
    "label_columns": label_columns,
    "selected_threshold": selected_threshold,
    "uses_sigmoid": True,
    "loss_function": "BCEWithLogitsLoss",
    "architecture": "mobilenet_v3_large",
    "best_val_top1_accuracy": best_val_top1_accuracy,
    "curated_top1_accuracy": curated_eval["top1_accuracy"],
    "curated_top1_macro_f1": curated_eval["top1_macro_f1"],
    "real_world_top1_accuracy": real_world_eval["top1_accuracy"],
    "real_world_top1_macro_f1": real_world_eval["top1_macro_f1"],
    "real_world_fallback_summary": real_world_fallback_summary
}

torch.save(checkpoint, multilabel_model_path)

print("Saved multi-label model to:", multilabel_model_path)
print("Model exists:", multilabel_model_path.exists())

Saved multi-label model to: ..\models\style_mobilenet_v3_large_multilabel_bce.pth
Model exists: True


## 28. Analyze Final Prediction Combinations

After applying the selected threshold with top-1 fallback, I inspect which style combinations the model predicts.

This helps check whether the multi-label model is producing realistic combinations, such as sporty + streetwear or gothic + streetwear.

In [24]:
print("Curated predicted style combinations:")
curated_combination_counts = (
    curated_prediction_df["predicted_styles_threshold_fallback"]
    .value_counts()
    .reset_index()
)

curated_combination_counts.columns = ["predicted_combination", "count"]
display(curated_combination_counts)

print("Real-world predicted style combinations:")
real_world_combination_counts = (
    real_world_prediction_df["predicted_styles_threshold_fallback"]
    .value_counts()
    .reset_index()
)

real_world_combination_counts.columns = ["predicted_combination", "count"]
display(real_world_combination_counts)

curated_combination_counts.to_csv(
    results_dir / "curated_predicted_combination_counts.csv",
    index=False
)

real_world_combination_counts.to_csv(
    results_dir / "real_world_predicted_combination_counts.csv",
    index=False
)

Curated predicted style combinations:


,predicted_combination,count
0,formal,26
1,gothic,26
2,sporty + streetwear,16
3,streetwear,15
4,sporty,14
5,formal + sporty,9
6,formal + streetwear,5
7,formal + gothic,2
8,gothic + streetwear,2
9,gothic + sporty,2


Real-world predicted style combinations:


,predicted_combination,count
0,streetwear,23
1,formal,14
2,sporty + streetwear,9
3,gothic,7
4,formal + streetwear,6
5,formal + sporty,6
6,sporty,5
7,gothic + streetwear,5
8,formal + gothic,3
9,gothic + sporty + streetwear,1


## 29. Final Comparison Table

This section compares the previous single-label MobileNetV3 model with the new multi-label MobileNetV3 model.

The top-1 accuracy shows whether the model still works as a normal classifier. The multi-label output behaviour shows whether it can provide combined aesthetic directions.

In [25]:
final_comparison = pd.DataFrame([
    {
        "model": "Previous single-label MobileNetV3",
        "training_setup": "CrossEntropyLoss + softmax",
        "curated_top1_accuracy": 0.8167,
        "curated_macro_f1": 0.8105,
        "real_world_top1_accuracy": 0.6250,
        "real_world_macro_f1": 0.6242,
        "selected_threshold": "not applicable",
        "real_world_zero_label_predictions": "not applicable",
        "real_world_multi_label_predictions": "not applicable",
        "real_world_avg_predicted_labels": "not applicable"
    },
    {
        "model": "New multi-label MobileNetV3",
        "training_setup": "BCEWithLogitsLoss + sigmoid",
        "curated_top1_accuracy": curated_eval["top1_accuracy"],
        "curated_macro_f1": curated_eval["top1_macro_f1"],
        "real_world_top1_accuracy": real_world_eval["top1_accuracy"],
        "real_world_macro_f1": real_world_eval["top1_macro_f1"],
        "selected_threshold": selected_threshold,
        "real_world_zero_label_predictions": real_world_fallback_summary["zero_label_predictions"],
        "real_world_multi_label_predictions": real_world_fallback_summary["multi_label_predictions"],
        "real_world_avg_predicted_labels": real_world_fallback_summary["avg_predicted_labels"]
    }
])

display(final_comparison)

final_comparison_path = results_dir / "final_multilabel_model_comparison.csv"

final_comparison.to_csv(final_comparison_path, index=False)

print("Saved final comparison to:", final_comparison_path)

,model,training_setup,curated_top1_accuracy,curated_macro_f1,real_world_top1_accuracy,real_world_macro_f1,selected_threshold,real_world_zero_label_predictions,real_world_multi_label_predictions,real_world_avg_predicted_labels
0,Previous single-label MobileNetV3,CrossEntropyLoss + softmax,0.8167,0.810500,0.625,0.624200,not applicable,not applicable,not applicable,not applicable
1,New multi-label MobileNetV3,BCEWithLogitsLoss + sigmoid,0.8000,0.796846,0.625,0.625186,0.35,0,31,1.4125


Saved final comparison to: ..\results\style_multilabel_training\final_multilabel_model_comparison.csv


## 30. Save Final Experiment Summary

The final summary stores the most important result of the experiment.

This makes it easier to use the outcome in the project documentation and portfolio evidence later.

In [26]:
final_experiment_summary = {
    "experiment": "17_fashion_style_multilabel_training",
    "model": "MobileNetV3 Large",
    "training_setup": "BCEWithLogitsLoss + sigmoid",
    "training_rows": len(train_labels_df),
    "single_label_training_rows": int((train_labels_df["label_count"] == 1).sum()),
    "multi_label_training_rows": int((train_labels_df["label_count"] > 1).sum()),
    "best_validation_top1_accuracy": best_val_top1_accuracy,
    "curated_top1_accuracy": curated_eval["top1_accuracy"],
    "curated_macro_f1": curated_eval["top1_macro_f1"],
    "real_world_top1_accuracy": real_world_eval["top1_accuracy"],
    "real_world_macro_f1": real_world_eval["top1_macro_f1"],
    "selected_threshold": selected_threshold,
    "curated_zero_label_predictions_after_fallback": curated_fallback_summary["zero_label_predictions"],
    "curated_multi_label_predictions_after_fallback": curated_fallback_summary["multi_label_predictions"],
    "curated_avg_predicted_labels_after_fallback": curated_fallback_summary["avg_predicted_labels"],
    "real_world_zero_label_predictions_after_fallback": real_world_fallback_summary["zero_label_predictions"],
    "real_world_multi_label_predictions_after_fallback": real_world_fallback_summary["multi_label_predictions"],
    "real_world_avg_predicted_labels_after_fallback": real_world_fallback_summary["avg_predicted_labels"],
    "main_result": (
        "The multi-label model did not improve real-world top-1 accuracy compared with the previous "
        "single-label MobileNetV3 model, but it produced usable multi-style outputs when using "
        "a 0.35 sigmoid threshold with top-1 fallback."
    ),
    "selected_for_application": True
}

final_experiment_summary_df = pd.DataFrame([final_experiment_summary])

final_summary_path = results_dir / "final_multilabel_training_summary.csv"

final_experiment_summary_df.to_csv(final_summary_path, index=False)

print("Saved final experiment summary to:", final_summary_path)

display(final_experiment_summary_df)

Saved final experiment summary to: ..\results\style_multilabel_training\final_multilabel_training_summary.csv


,experiment,model,training_setup,training_rows,single_label_training_rows,multi_label_training_rows,best_validation_top1_accuracy,curated_top1_accuracy,curated_macro_f1,real_world_top1_accuracy,real_world_macro_f1,selected_threshold,curated_zero_label_predictions_after_fallback,curated_multi_label_predictions_after_fallback,curated_avg_predicted_labels_after_fallback,real_world_zero_label_predictions_after_fallback,real_world_multi_label_predictions_after_fallback,real_world_avg_predicted_labels_after_fallback,main_result,selected_for_application
0,17_fashion_style_multilabel_training,MobileNetV3 Large,BCEWithLogitsLoss + sigmoid,660,604,56,0.8,0.8,0.796846,0.625,0.625186,0.35,0,39,1.35,0,31,1.4125,The multi-label model did not improve real-wor...,True


## 31. Final Conclusion

This experiment tested whether the style classifier could be improved by changing from single-label classification to multi-label classification.

The model was trained using `BCEWithLogitsLoss` instead of `CrossEntropyLoss`, and sigmoid scores were used instead of softmax probabilities. This allowed the model to treat each style independently instead of forcing every image into one final aesthetic class.

The top-1 results show that the new multi-label model performs similarly to the previous single-label MobileNetV3 model. On the real-world test set, both models reached 0.6250 accuracy. This means the multi-label setup did not improve the main style prediction accuracy.

However, the multi-label behaviour is useful for the final application. With a sigmoid threshold of 0.35 and top-1 fallback, the model produced no empty predictions and predicted multiple styles for 31 out of 80 real-world test images. This is useful because fashion items often combine aesthetics, especially sporty + streetwear, gothic + streetwear, and formal + streetwear.

The main conclusion is that multi-label training is justified for recommendation behaviour, but not because it improves top-1 accuracy. Its value is that it gives the recommendation system more flexible style information. The model can now support combined aesthetic directions instead of only returning one forced style.